# 🗞️ Hacker News Digest Bot
### End-to-End NLP / LLM / RAG Project

---

## 🗺️ What You're Building

A fully automated daily news digest pipeline that:
1. **Fetches** top stories from Hacker News API (async)
2. **Scrapes** full article text from each URL
3. **Classifies** each article by topic (AI, crypto, politics, science)
4. **Summarizes** each article into 3 key sentences
5. **Answers** your natural language questions using RAG + Claude API

```
[HN API] → fetch IDs → fetch metadata (async)
         → scrape article text (rate-limited)
         → classify topic (weighted keywords)
         → extractive summary (sentence scoring)
         → embed articles (sentence-transformers)
         → RAG: find relevant articles (cosine similarity)
         → LLM answers your question (Claude API)
```

---

## 📚 Concepts You Will Learn

| Stage | Concept |
|-------|---------|
| 1 | Async programming, aiohttp, asyncio.gather |
| 2 | Web scraping, HTML text extraction, trafilatura |
| 3 | Bag-of-words, keyword classification, weighted scoring |
| 4 | Extractive summarization, stopwords, sentence scoring |
| 5 | Embeddings, cosine similarity, RAG, LLM API calls |

---
## ⚙️ Step 0 — Install Dependencies

Run this cell first. It installs everything the project needs.

| Library | Why |
|---------|-----|
| `aiohttp` | Async HTTP client (replaces `requests` for concurrent fetching) |
| `trafilatura` | Extracts readable article text from raw HTML |
| `sentence-transformers` | Turns sentences into meaning vectors (embeddings) |
| `scikit-learn` | Cosine similarity calculation |
| `anthropic` | Claude API client |
| `nest_asyncio` | Allows `asyncio` to run inside Jupyter's event loop |

In [ ]:
!pip install -q aiohttp trafilatura sentence-transformers scikit-learn anthropic nest_asyncio

---
## 🔑 Step 0.1 — Set Your Anthropic API Key

Get your free key at 👉 https://console.anthropic.com → API Keys

> **Tip:** In Colab you can also use `Secrets` (🔑 icon in left sidebar) to store your key safely as `ANTHROPIC_API_KEY`.

In [ ]:
import os

# Option 1: Set directly (don't commit this to GitHub!)
os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"

# Option 2: Use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

print("✅ API key set.")

---
## 📦 Step 0.2 — Imports & Global Config

In [ ]:
import asyncio
import aiohttp
import trafilatura
import numpy as np
import anthropic
import nest_asyncio
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Jupyter already runs an event loop — nest_asyncio patches it
# so we can use asyncio.run() or await at the top level
nest_asyncio.apply()

# ── Global Config ─────────────────────────────────────────────────────────────
TOP_STORIES_URL    = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL_TEMPLATE  = "https://hacker-news.firebaseio.com/v0/item/{}.json"
ARTICLE_FETCH_LIMIT = 3   # Max concurrent article fetches (Semaphore)
TOP_N_STORIES      = 10   # How many HN stories to fetch

print("✅ All imports successful.")

---
# 🌐 Stage 1 & 2 — Async Data Collection + Article Scraping

## Concept: Synchronous vs Asynchronous

**Synchronous** (what `requests` does):
```
fetch story 1 → waiting... → done
fetch story 2 →             waiting... → done
fetch story 3 →                          waiting... → done
```
Each request blocks the thread until the server responds.

**Asynchronous** (what `aiohttp` does):
```
fetch story 1 → waiting...
fetch story 2 →   waiting...
fetch story 3 →     waiting...
                        ↓ results come back as they arrive
```
All requests run concurrently on a single thread. While one waits for the server, the others keep going.

## Concept: asyncio.Semaphore
A Semaphore limits how many coroutines can run at the same time. Without it, fetching 10 articles simultaneously may trigger **rate limiting (HTTP 429)** on target websites.

📖 **Read more:** https://realpython.com/async-io-python/

In [ ]:
# ── Stage 1: Fetch a single HN story's metadata ───────────────────────────────
async def fetch_item(session, item_id):
    """
    Fetches metadata (title, url, score, etc.) for one HN story.
    Returns a dict like: {'title': '...', 'url': '...', 'score': 42, ...}
    """
    url = ITEM_URL_TEMPLATE.format(item_id)
    async with session.get(url) as response:
        response.raise_for_status()
        return await response.json()


# ── Stage 2: Fetch + extract readable text from an article URL ────────────────
async def fetch_article_text(session, url, semaphore):
    """
    Downloads the HTML of an article URL and extracts clean readable text.

    trafilatura is used instead of BeautifulSoup because it:
    - Strips ads, nav bars, footers, and other clutter automatically
    - Works well across varied HTML structures (news, blogs, wikis)
    - Is actively maintained with better extraction quality

    The Semaphore ensures only ARTICLE_FETCH_LIMIT fetches run at once,
    preventing rate-limiting from target websites.
    """
    async with semaphore:  # Only N coroutines enter this block at once
        try:
            timeout = aiohttp.ClientTimeout(total=10)
            async with session.get(url, timeout=timeout) as response:
                response.raise_for_status()
                html = await response.text()

            # trafilatura extracts main article body from raw HTML
            text = trafilatura.extract(html)
            return text if text else "No readable text found."

        except Exception as e:
            return f"Failed to fetch: {e}"


print("✅ Stage 1 & 2 functions defined.")

---
# 🏷️ Stage 3 — Topic Classification

## Concept: Bag of Words
The simplest NLP classification: treat text as an unordered collection ("bag") of words. Count how many topic-related words appear. The topic with the highest count wins.

## Concept: Weighted Keywords
Not all keywords are equally trustworthy:
- `"ai"` can appear inside `"main"`, `"again"`, `"paid"` → **low weight (1)**
- `"neural network"` almost never appears unless the article is genuinely about AI → **high weight (3)**

Longer, more specific phrases get higher weights.

## What's Next After This?
Keyword matching has limits — it can't handle synonyms or context. The natural upgrade is **TF-IDF** (Term Frequency–Inverse Document Frequency), which scores words by how unique they are across documents.

📖 **Read more:** https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction

In [ ]:
# ── Stage 3: Weighted Keyword Topic Classifier ────────────────────────────────
def classify_topic(text):
    """
    Classifies article text into a topic using weighted keyword matching.

    Each keyword is a (phrase, weight) tuple:
    - weight 3 = highly specific phrase, very trustworthy signal
    - weight 2 = moderately specific
    - weight 1 = common word, weak signal

    Returns: 'AI' | 'crypto' | 'politics' | 'science' | 'other'
    """
    text = text.lower()

    topic_keywords = {
        "AI": [
            ("artificial intelligence", 3), ("machine learning", 3),
            ("neural network", 3),          ("deep learning", 3),
            ("llm", 2),                      ("gpt", 2),
            ("openai", 2),                   ("model", 1),
            ("ai", 1),
        ],
        "crypto": [
            ("blockchain", 3),    ("decentralized", 3),
            ("ethereum", 2),      ("bitcoin", 2),
            ("defi", 2),          ("crypto", 1),
            ("token", 1),         ("nft", 1),
            ("wallet", 1),
        ],
        "politics": [
            ("election", 3),      ("congress", 3),
            ("legislation", 3),   ("senate", 2),
            ("president", 2),     ("minister", 2),
            ("government", 1),    ("policy", 1),
            ("political", 1),
        ],
        "science": [
            ("peer-reviewed", 3), ("experiment", 3),
            ("clinical trial", 3),("physics", 2),
            ("biology", 2),       ("chemistry", 2),
            ("research", 1),      ("study", 1),
            ("scientists", 1),
        ],
    }

    # Score each topic by summing weights of matched keywords
    scores = {topic: 0 for topic in topic_keywords}
    for topic, keywords in topic_keywords.items():
        for keyword, weight in keywords:
            if keyword in text:
                scores[topic] += weight

    best_topic = max(scores, key=scores.get)
    return best_topic if scores[best_topic] > 0 else "other"


# Quick sanity check
test = "OpenAI released a new large language model with improved deep learning capabilities."
print(f"Test classification: '{classify_topic(test)}'  (expected: AI)")
print("✅ Stage 3 function defined.")

---
# ✂️ Stage 4 — Extractive Summarization

## Concept: Two Types of Summarization

| Type | How it works | Needs ML? |
|------|-------------|----------|
| **Extractive** | Pick the most important *existing* sentences from the text | ❌ No |
| **Abstractive** | *Generate new sentences* that capture the meaning | ✅ Yes (LLM) |

We start with extractive — it's fast, always grammatically correct, and never hallucinates.

## How It Works
1. Split text into sentences
2. Remove **stopwords** ("the", "a", "is" — words that carry no meaning)
3. Score each sentence by counting remaining meaningful words
4. Return the top N highest-scoring sentences in original order

## Weakness
A sentence like *"This is a very important and significant development"* scores high (many non-stopword words) but carries little information. That's why TF-IDF or embeddings give better results in production.

📖 **Read more:** https://towardsdatascience.com/understand-text-summarization-and-create-your-own-summarizer-in-python-b26a9f09fc70

In [ ]:
# ── Stage 4: Extractive Summarizer ────────────────────────────────────────────
def summarize_extractive(text, n=3):
    """
    Returns the n most 'important' sentences from the text.

    Importance = number of non-stopword words in the sentence.
    Sentences are returned in their original order (not by score)
    to preserve the natural narrative flow.
    """
    # Common words that carry almost no meaning
    stopwords = {
        "the", "a", "an", "is", "it", "in", "on", "at", "to",
        "and", "or", "of", "for", "this", "that", "was", "are",
        "with", "by", "from", "as", "be", "has", "have", "had"
    }

    # Split into sentences, ignore very short ones (likely fragments)
    sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 20]
    if not sentences:
        return text[:300]

    # Score each sentence by counting meaningful (non-stopword) words
    scores = {}
    for sentence in sentences:
        words = sentence.lower().split()
        score = sum(1 for word in words if word not in stopwords)
        scores[sentence] = score

    # Pick the top n sentences
    top_sentences = sorted(scores, key=scores.get, reverse=True)[:n]

    # Return in original order to preserve flow
    ordered = [s for s in sentences if s in top_sentences]
    return ". ".join(ordered) + "."


# Quick sanity check
test_text = "The sky is blue. Scientists discovered a new black hole 1000 light years away using advanced telescope technology. It was a nice day. The black hole has a mass 50 times larger than our sun and emits powerful gravitational waves."
print("Summary:", summarize_extractive(test_text, n=2))
print("✅ Stage 4 function defined.")

---
# 🤖 Stage 5 — RAG (Retrieval-Augmented Generation)

## Concept: What is RAG?
**RAG = Retrieval-Augmented Generation**

Instead of asking an LLM from memory:
1. **Retrieve** the most relevant documents from your own data
2. **Augment** the LLM prompt by injecting those documents as context
3. **Generate** an answer grounded in real, current information

This solves a core LLM problem: models are frozen at their training cutoff. RAG lets them answer questions about *today's news*, your private documents, or any live data.

## Concept: Embeddings
An embedding turns a sentence into a **list of numbers (vector)** that encodes its *meaning*.

- `"quantum computing"` → `[0.12, -0.45, 0.88, ...]`  (384 numbers)
- `"qubit superposition"` → `[0.11, -0.43, 0.85, ...]`  (very close!)
- `"bitcoin price"` → `[0.91, 0.22, -0.13, ...]`  (far away)

Similar meanings = vectors pointing in the same direction in space.

## Concept: Cosine Similarity
Measures the angle between two vectors. Score of **1.0** = identical meaning. Score of **0.0** = completely unrelated.

We embed the user's question, embed all articles, then find which articles are *closest* in meaning.

📖 **Embeddings:** https://www.sbert.net/docs/sentence_transformer/pretrained_models.html  
📖 **Cosine similarity:** https://developers.google.com/machine-learning/clustering/similarity/measuring-similarity  
📖 **RAG explained:** https://www.pinecone.io/learn/retrieval-augmented-generation/

In [ ]:
# ── Load embedding model (downloads ~90MB on first run) ───────────────────────
# all-MiniLM-L6-v2 is a fast, small model producing 384-dimensional embeddings
# It's a great balance of speed and quality for semantic search
print("Loading embedding model... (first run downloads ~90MB)")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded.")

In [ ]:
# ── Stage 5a: Find the N most relevant articles for a question ────────────────
def find_relevant_articles(question, stories, texts, n=3):
    """
    Uses semantic search (embeddings + cosine similarity) to find
    the most relevant articles for a given question.

    Unlike keyword search, this understands *meaning* — so
    'quantum computing' will match articles about 'qubits'
    even if they share no exact words.

    Args:
        question: User's natural language question
        stories:  List of HN story metadata dicts
        texts:    List of article text strings (same order as stories)
        n:        Number of top articles to return

    Returns: List of (story_dict, article_text, similarity_score) tuples
    """
    # Embed the question into a 384-dim vector
    question_embedding = embedding_model.encode([question])

    # Embed all article texts
    text_embeddings = embedding_model.encode(texts)

    # Cosine similarity: question vs every article → array of scores
    scores = cosine_similarity(question_embedding, text_embeddings)[0]

    # Sort by score descending, take top N
    top_indices = np.argsort(scores)[::-1][:n]
    return [(stories[i], texts[i], scores[i]) for i in top_indices]


# ── Stage 5b: Build a RAG prompt and ask Claude ───────────────────────────────
def ask_about_news(question, stories, texts):
    """
    Full RAG pipeline:
    1. Retrieve most relevant articles via semantic search
    2. Build a prompt that injects those articles as context
    3. Ask Claude to answer based ONLY on that context

    Grounding the LLM in real articles prevents hallucination
    and ensures answers reflect today's actual news.
    """
    # Step 1: Retrieve
    relevant = find_relevant_articles(question, stories, texts)

    # Step 2: Build context block from top articles
    context = ""
    for story, text, score in relevant:
        title = story.get("title", "No Title")
        context += f"Article: {title}\n{text[:500]}\n\n"

    # Step 3: RAG prompt — instruct Claude to stay grounded in context
    prompt = f"""You are a news assistant. Answer the question based ONLY on the articles below.
Do not use outside knowledge. If the articles don't contain the answer, say so.

Articles:
{context}

Question: {question}
Answer:"""

    # Step 4: Call Claude API
    # anthropic.Anthropic() reads ANTHROPIC_API_KEY from environment automatically
    client = anthropic.Anthropic()
    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text


print("✅ Stage 5 functions defined.")

---
# 🚀 Full Pipeline — All Stages Combined

This is the main function. It runs all 5 stages end-to-end.

In [ ]:
# ── Main Pipeline: All 5 Stages ───────────────────────────────────────────────
async def get_top_hn_stories(limit=TOP_N_STORIES):
    semaphore = asyncio.Semaphore(ARTICLE_FETCH_LIMIT)

    async with aiohttp.ClientSession() as session:

        # ── Stage 1a: Fetch top story IDs ─────────────────────────────────────
        print("📡 Fetching top story IDs from HN...")
        async with session.get(TOP_STORIES_URL) as response:
            response.raise_for_status()
            story_ids = await response.json()
        print(f"   Got {len(story_ids)} IDs. Taking top {limit}.")

        # ── Stage 1b: Fetch all metadata concurrently (asyncio.gather) ────────
        # asyncio.gather(*tasks) runs all coroutines concurrently and waits
        # for all of them to finish. The * unpacks the list into arguments.
        print("\n📰 Fetching story metadata concurrently...")
        stories = await asyncio.gather(*[
            fetch_item(session, sid) for sid in story_ids[:limit]
        ])
        print(f"   Fetched metadata for {len(stories)} stories.")

        # ── Stage 2: Fetch article text (rate-limited by Semaphore) ──────────
        print(f"\n📄 Scraping article text (max {ARTICLE_FETCH_LIMIT} concurrent)...")
        article_texts = await asyncio.gather(*[
            fetch_article_text(session, story.get("url"), semaphore)
            if story.get("url")
            else asyncio.coroutine(lambda: "No URL.")()  # Fallback for self-posts
            for story in stories
        ])
        print(f"   Scraped text for {len(article_texts)} articles.")

    # ── Stage 3 + 4: Classify & Summarize, then print digest ─────────────────
    print(f"\n{'='*65}")
    print(f"  🗞️  HACKER NEWS DIGEST — TOP {limit} STORIES")
    print(f"{'='*65}\n")

    for i, (story, text) in enumerate(zip(stories, article_texts), 1):
        title   = story.get("title", "No Title")
        url     = story.get("url", "No URL (self-post)")
        score   = story.get("score", 0)
        topic   = classify_topic(text)          # Stage 3
        summary = summarize_extractive(text)    # Stage 4

        print(f"{i:2}. [{topic.upper():10}] {title}")
        print(f"    🔗 {url}")
        print(f"    ⭐ Score: {score}")
        print(f"    📝 {summary[:300]}")
        print(f"    {'─'*60}")

    # ── Stage 5: RAG — Ask a natural language question ────────────────────────
    all_stories = list(stories)
    all_texts   = list(article_texts)

    # Change this question to ask anything about today's HN news
    question = "What is happening with AI today?"

    print(f"\n{'='*65}")
    print(f"  🤖 RAG — ASKING CLAUDE ABOUT TODAY'S NEWS")
    print(f"{'='*65}")
    print(f"\n❓ Question: {question}")
    print("\n🔍 Finding relevant articles via semantic search...")

    # Show which articles were retrieved
    relevant = find_relevant_articles(question, all_stories, all_texts)
    for story, text, sim_score in relevant:
        print(f"   • [{sim_score:.2f}] {story.get('title', 'No Title')}")

    print("\n💬 Asking Claude...")
    answer = ask_about_news(question, all_stories, all_texts)

    print(f"\n💡 Answer:\n{answer}")
    print(f"\n{'='*65}")


# ── Run it ────────────────────────────────────────────────────────────────────
await get_top_hn_stories()

---
# 🧪 Bonus: Ask Your Own Questions

Re-run the RAG step with any question you want about today's news.

In [ ]:
# ── Ask your own question ─────────────────────────────────────────────────────
# Make sure you've already run the full pipeline above so
# `stories` and `article_texts` are loaded in memory.

my_question = "Are there any security or privacy concerns in today's news?"

print(f"❓ Question: {my_question}\n")
print("🔍 Relevant articles found:")
for story, text, score in find_relevant_articles(my_question, stories, article_texts):
    print(f"   • [{score:.2f}] {story.get('title')}")

print("\n💡 Answer:")
print(ask_about_news(my_question, stories, article_texts))